In [1]:
#Автоматическая перезагрузка при изменениях
%load_ext autoreload
%autoreload 2

In [2]:
# Для MPS рекомендуется включить fallback на CPU для неподдерживаемых операций
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [ ]:
from src.data_utils import load_and_clear_data, load_or_tokenize, create_sequences, create_data_split, build_vocab
from src.constants import BATCH_SIZE
from src.next_token_dataset import NextTokenDataset
from src.lstm_model import LSTMLanguageModel
from src.lstm_train import train_model
from src.lstm_test import test_model
from src.generate_examples_lstm import generate_examples

import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

### Подготовка данных

In [4]:
texts = load_and_clear_data()

Загружено 10000 текстов, после очистки осталось 9979 текстов.
Примеры очищенных текстов: ['a thats a bummer. you shoulda got david carr of third day to do it. d', 'is upset that he cant update his facebook by texting it... and might cry as a result school today also. blah!', 'i dived many times for the ball. managed to save 50 the rest go out of bounds', 'my whole body feels itchy and like its on fire', 'no, its not behaving at all. im mad. why am i here? because i cant see you all over there.']


In [5]:
tokenized_texts = load_or_tokenize(texts)

Загрузка токенизированных данных из data/tokenized_texts.pkl...


In [6]:
X, Y = create_sequences(tokenized_texts)

In [7]:
display(X[1:2])
display(Y[1:2])

[['a',
  'thats',
  'a',
  'bummer',
  '.',
  'you',
  'shoulda',
  'got',
  'david',
  'carr',
  'of',
  'third',
  'day',
  'to',
  'do',
  'it',
  '.',
  'd']]

[['thats',
  'a',
  'bummer',
  '.',
  'you',
  'shoulda',
  'got',
  'david',
  'carr',
  'of',
  'third',
  'day',
  'to',
  'do',
  'it',
  '.',
  'd',
  '<EOS>']]

In [8]:
X_train, Y_train, X_val, Y_val, X_test, Y_test = create_data_split(X, Y) 

Train: 127046, Val: 15881, Test: 15881


In [9]:
lengths = [len(tokens) for tokens in tokenized_texts]
print(f"Средняя длина: {sum(lengths) / len(lengths):.1f}")
print(f"Медиана: {sorted(lengths)[len(lengths)//2]}")
print(f"90-й перцентиль: {sorted(lengths)[int(len(lengths)*0.9)]}")

Средняя длина: 16.9
Медиана: 16
90-й перцентиль: 28


In [10]:
# Создание словаря
word2idx, idx2word = build_vocab(tokenized_texts)

vocab_size = len(word2idx)
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 13784


In [11]:
# Создание датасетов
train_dataset = NextTokenDataset(X_train, Y_train, word2idx)
val_dataset = NextTokenDataset(X_val, Y_val, word2idx)
test_dataset = NextTokenDataset(X_test, Y_test, word2idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

### Тренировка модели LSTM

In [12]:
device = (
        "cuda" if torch.cuda.is_available() else
        "mps" if torch.backends.mps.is_available() else
        "cpu"
    )
print(f"Using device: {device}")

print("torch.backends.mps.is_available:", torch.backends.mps.is_available())
print("torch.backends.mps.is_built:", torch.backends.mps.is_built())

Using device: mps
torch.backends.mps.is_available: True
torch.backends.mps.is_built: True


In [13]:
model = LSTMLanguageModel(vocab_size, word2idx, idx2word).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [14]:
train_losses, val_losses = train_model(model, train_loader, val_loader, criterion, optimizer)

Epoch 1/10 [Train]: 100%|██████████| 3971/3971 [03:08<00:00, 21.09it/s, loss=3.9658]


Epoch 1, Train Loss: 4.6439, Val loss: 3.4346


Epoch 2/10 [Train]: 100%|██████████| 3971/3971 [03:14<00:00, 20.39it/s, loss=2.9082]


Epoch 2, Train Loss: 3.2198, Val loss: 2.7650


Epoch 3/10 [Train]: 100%|██████████| 3971/3971 [03:15<00:00, 20.36it/s, loss=2.6705]


...

Epoch 8, Train Loss: 2.3318, Val loss: 2.1979


Epoch 9/10 [Train]: 100%|██████████| 3971/3971 [03:18<00:00, 20.03it/s, loss=2.0281]


Epoch 9, Train Loss: 2.3061, Val loss: 2.1853


Epoch 10/10 [Train]: 100%|██████████| 3971/3971 [03:20<00:00, 19.84it/s, loss=3.4070]


Epoch 10, Train Loss: 2.2893, Val loss: 2.1788


In [ ]:
test_model(model, test_loader, idx2word, word2idx, device)

In [ ]:
# Разнообразные фразы
seed_phrases = [
    "i feel",
    "i am feeling so",
    "i love",
    "today is a",
    "i hate when",
    "so sad",
    "i just want"
]

# Генерация примеров
generate_examples(
    model=model, 
    seed_texts=seed_phrases, 
    word2idx=word2idx, 
    idx2word=idx2word, 
    max_length=10,
    temperature=0.8
)


📝 ПРИМЕРЫ ГЕНЕРАЦИИ МОДЕЛИ

🌱 Seed: "i feel"
🤖 Output: i feel a loser ! not gon na go back
------------------------------------------------------------

🌱 Seed: "i love"
🤖 Output: i love the code .
------------------------------------------------------------

🌱 Seed: "today is"
🤖 Output: today is going to come to sleep . ugh . no fun
------------------------------------------------------------

🌱 Seed: "i hate"
🤖 Output: i hate it too .
------------------------------------------------------------

🌱 Seed: "so sad"
🤖 Output: so sad
------------------------------------------------------------


In [17]:
eval_lstm(model, test_loader, device)

NameError: name 'eval_lstm' is not defined